# Readmission or death after discharge, with exact event times: continuous-time survival TMLE

## At a glance

A hospital offers some patients navigation at discharge, more often those with a higher
readmission-risk score. Follow-up records the exact time of each patient's first unplanned
readmission, of death before any readmission, or of leaving follow-up. Does the offer change the
chance of each event within three months?

Had every patient stayed in follow-up, offering navigation to everyone rather than to no one would
change the risk of readmission by month 3 by -0.160 (95% confidence interval -0.225 to -0.094), about
16 fewer patients readmitted per 100. The data cannot tell whether it changes death by month 3:
that interval runs from -0.070 to 0.060. Over the first three months, the offer adds 0.407 months
free of both events (0.249 to 0.565), about 12 days. These answers are causal only if the risk score
holds every common cause of the offer, of the two events and of leaving follow-up, which the data
cannot check.

The data are simulated, so we know the true changes (-0.213 for readmission, -0.059 for death,
0.556 months), and each interval contains its value.

## The question

The program decides on the offer at discharge, which is time zero: follow-up starts there.
Readmission and death are competing events: a patient who dies can no longer be readmitted. So
each event's risk is the share of patients who have that event first, in the world as it is, where
neither event is removed. The target is each plan's risk of each event by month 1.5 and by month 3,
had every patient been offered navigation, or none, and had every patient stayed in follow-up. A second target is the
average time free of both events over the first three months, called the restricted mean survival
time.

```text
  risk_score ──▶ offered ──▶ readmission, death
       └──────────────────▶ readmission, death
  leaving follow-up, in this simulation: no arrow from the risk score, the offer or either event
  (the method also allows arrows from the risk score and the offer)
  assumed absent: an unrecorded cause of the offer and an event, or of leaving and an event
```

Patients with a higher risk score were offered navigation more often, were readmitted more often
and, in this simulation, died less often before a readmission, so the score is a confounder, a variable that drives both
the offer and the outcome. A patient who leaves follow-up hides any later event: that is
censoring. The data record each event's exact time. Methods for binned data count events only on
fixed check days, such as day 30, 60 and 90; this one uses each exact time, so no event is moved to
a later check, and the risks can be read at any times you declare before the fit.

## The data

`make_exact_readmission` draws 1,000 simulated patients and returns the true answers, which real
data never give.

In [1]:
import numpy as np
import pandas as pd

import cleverly
from cleverly import plotting

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.2.8


In [2]:
from cleverly.datasets import EXACT_CAUSES, make_exact_readmission

# With your own data, load one row per patient, such as pd.read_csv(...), with the risk score,
# the offer, the time of the first event or of leaving follow-up, and an event code: 1 for a
# readmission, 2 for a death before any readmission, 0 for leaving follow-up first.
# Real data have no `truth`: delete the lines marked "simulation only" here and below.
frame, truth = make_exact_readmission(n=1_000, seed=2026)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().round(3).to_string())
print()
labels = {0: "left follow-up", 1: "readmission", 2: "death"}
print("what ended each patient's follow-up:", frame["event"].map(labels).value_counts().to_dict())
print("patients offered navigation:", int(frame["offered"].sum()))
print()

# simulation only, to the end of the cell: the true risk of each event by each month.
known = pd.DataFrame(
    [
        {
            "plan": plan,
            "event": cause,
            "month": month,
            "true risk": truth[f"cif_regimen[{plan}, {cause} @ t={month:g}]"],
        }
        for plan in ("offered", "not offered")
        for cause in EXACT_CAUSES.values()
        for month in (1.5, 3.0)
    ]
)
print("true risk had every patient stayed in follow-up:")
print(known.round(4).to_string(index=False))

rows and columns: (1000, 4)
['time', 'event', 'offered', 'risk_score']
    time  event  offered  risk_score
0  7.327      2        0      -0.793
1  1.405      1        0       0.241
2  2.216      2        0      -1.896
3  1.105      2        0       1.396
4  0.085      1        1       0.638

what ended each patient's follow-up: {'readmission': 358, 'left follow-up': 354, 'death': 288}
patients offered navigation: 510

true risk had every patient stayed in follow-up:
       plan       event  month  true risk
    offered readmission    1.5     0.1213
    offered readmission    3.0     0.2126
    offered       death    1.5     0.1213
    offered       death    3.0     0.2126
not offered readmission    1.5     0.2782
not offered readmission    3.0     0.4251
not offered       death    1.5     0.1740
not offered       death    3.0     0.2712


Each row is one patient. `time` is the months from discharge to the event that ended follow-up,
and `event` says which: 1 a readmission, 2 a death before any readmission, 0 leaving follow-up
first. The true risks are those of patients who all stayed in follow-up. Under the offer, the two
true risks happen to be equal: in this simulation the offer turns the readmission rate into the
mirror image of the death rate, and the risk score is symmetric around zero.

## Why a simple comparison misleads

In [3]:
means = frame.groupby("offered")["risk_score"].mean()
print("mean risk score: offered", round(means[1], 3), "| not offered", round(means[0], 3))
print()

# The simple comparison: the share with each recorded event by month 3, offered minus not.
print("share with each recorded event by month 3, offered minus not offered:")
for code, cause in EXACT_CAUSES.items():
    recorded = frame["event"].eq(code) & frame["time"].le(3.0)
    shares = recorded.groupby(frame["offered"]).mean()
    line = f"  {cause:12s} simple {shares[1] - shares[0]:+.4f}"
    true = truth[f"ate_regimen[offered vs not offered, {cause} @ t=3]"]  # simulation only
    line += f", true {true:+.4f}"  # simulation only
    print(line)
left = frame["event"].eq(0) & frame["time"].le(3.0)
print("share who left follow-up by month 3:", round(left.mean(), 3))

mean risk score: offered 0.458 | not offered -0.433

share with each recorded event by month 3, offered minus not offered:
  readmission  simple -0.0387, true -0.2125
  death        simple -0.1084, true -0.0586
share who left follow-up by month 3: 0.241


Offered patients had a higher risk score, which raises readmission and lowers death. So the
simple readmission difference, -0.0387, is far smaller in size than the true -0.2125, and the
simple death difference, -0.1084, is larger than the true -0.0586. The simple shares also count
the 0.241 of patients who left follow-up by month 3 as having had neither event, which moves both
differences too.

## Set up the analysis

Write the study description (`StudyProtocol`): who is in it, when follow-up starts, what each plan
means, and how death and leaving follow-up count. It does not change the estimate; each result
keeps a copy, so anyone reading a saved result knows what it answers. `exact_readmission_protocol`
adapts the program's description to this question: it changes when follow-up starts, the plans and
their versions, the outcome, the horizon, how death and leaving follow-up count, and the reasons
for the conditions. [Write down the study
description](../user-guide/data-design.md#write-down-the-study-description) lists the fields.

In [4]:
from cleverly.datasets import exact_readmission_protocol

# For your own study, build cleverly.StudyProtocol(...) directly, with the fields printed below.
protocol = exact_readmission_protocol()
print("\n".join(protocol.summary_lines()))

Population: Adults discharged home from a participating hospital during the enrollment period
Eligibility: Age 18 years or older; Discharged alive; Discharged home from a participating hospital
Follow-up starts: Hospital discharge, when the risk score is recorded and the offer is made
Strategies: Offer navigation to every patient at discharge; Offer navigation to no patient
Versions: The declared discharge navigation contacts; Usual discharge support without navigation contacts
Outcome: First unplanned readmission, and death before any readmission
Horizon: 3 months after discharge, with risks read at 1.5 and 3 months
Intercurrent events: Treat death before readmission as a competing event: report the risk of each event, with neither prevented; Treat leaving follow-up as censoring, so the target is the risk had every patient stayed in follow-up (hypothetical strategy); Analyze each navigation offer regardless of completed contacts
Interference unit: Individual patient
Rationale: The ris

Next, name the columns for the time, the event code and the offer (`TimeToEvent`), and the event
each code stands for (`causes=`). `grid="continuous"` uses each exact time, with no check days.
`RegimeContrast` asks for each plan's risks and their differences at `horizons=`, and for the mean
time free of both events up to each cut-off in `restricted_horizons=`. `cleverly` calls a plan a
regimen (regime in some output) and a patient a unit. `study.identify(...)` states the question and
lists the conditions an answer rests on.

In [5]:
from cleverly import CausalStudy, RegimeContrast, TimeToEvent

# For your own data, change the column names and the event labels.
design = TimeToEvent(
    time="time",
    event="event",
    treatment="offered",
    baseline=("risk_score",),
    grid="continuous",
    causes={1: "readmission", 2: "death"},
)
study = CausalStudy(frame, design=design, protocol=protocol)
question = study.identify(
    RegimeContrast(
        {"offered": 1, "not offered": 0},
        reference="not offered",
        horizons=(1.5, 3.0),
        restricted_horizons=(3.0,),
    )
)
print(question.summary())

                         Causal question (ate_regime)
Question:   contrast of each regime     Rows:                            1,000
            against the reference       Answered by:     adjusting for history
Outcome:    event (event code), timed
            by time
Treatment:  offered
Baseline:   risk_score
What each treatment time adjusts for
treatment   adjusted for
------------------------------------------------------------------------------
offered     risk_score
Conditions the answer rests on
condition                            checkable   what it says
------------------------------------------------------------------------------
well-defined treatment               no          Each unit's observed outcomes
                                                 are the ones it would have
                                                 had under the treatments it
                                                 actually received over time.
                                         

The four conditions in the table, read for this study, cover leaving follow-up too:

- no hidden confounding: the risk score holds every common cause of the offer and of either event,
  and leaving follow-up depends only on the offer and the risk score (this needs judgment);
- overlap: every kind of patient had a real chance of being offered navigation and of not being
  offered it, and some chance of staying in follow-up through month 3 (the data can partly check
  this from the fitted models, in "Check the fit", which also runs the checks the notes name);
- a well-defined treatment and no interference: the offer is one well-defined action, and one
  patient's offer does not change another's events (this needs judgment).

The methods table lists the one method that fits this question.

The risk of each event with neither prevented needs no condition on how readmission and death
depend on each other. "Go deeper" shows a question that does.

## Estimate each event's risk

The method, targeted maximum likelihood estimation (TMLE) for exact event times, models each
event's rate over time, called a hazard, and the rate of leaving follow-up. It then nudges the
predicted hazards in small steps, using the models of who got the offer and who stayed in
follow-up, until the remaining bias the models leave in each reported number is removed to first
order. That is what makes a simple interval valid (Rytgaard and van der Laan, 2024). The settings:

- `horizons=(1.5, 3.0)` are the times to read each risk. The `support` column below depends on the
  share of recorded events that had happened by each time, printed first: a simulation study checked risks only at times inside a range of it ([the
  continuous-time study](../technical-reference/method-evidence/continuous-time-survival.md)).
- `restricted_horizons=(3.0,)` declares the cut-off of the mean time free of both events. The fit
  adjusts every number it reports, so a cut-off must be declared before it runs.
- With no models named, each hazard is chosen by cross-validation from three Cox regressions (the
  offer alone, the offer and the score, and their product added), and the offer from logistic
  regressions. The rows are split into ten parts, and each patient's predictions come from models
  fitted on the other nine, so no model predicts a patient it learned from.
- `random_state=0` fixes the splits.

In [ ]:
result = question.estimate(random_state=0)

# The support labels read the share of recorded events that had happened by each time.
events = frame.loc[frame["event"].gt(0), "time"]
shares = [round(float(events.le(month).mean()), 3) for month in (1.5, 3.0)]
print("share of recorded events by month 1.5 and month 3:", shares)
print()

table = result.to_frame()
table = table[table["estimand"].str.match(r"(cif|ate)_regimen")]
table = table[["estimand", "psi", "ci_lower", "ci_upper", "support"]].copy()
table["true"] = table["estimand"].map(truth)  # simulation only
print(table.round(3).to_string(index=False))
print()
print("band over all twelve risks and differences:", result.simultaneous)
risk_fig = plotting.curve(result, time_label="Months since discharge")

`psi` is the estimate. `cif_regimen[offered, readmission @ t=3]` is the share readmitted by month 3
had every patient been offered navigation: 0.245, against 0.405 with no offer. `ate_regimen[...]`
is the difference between the two plans. Each support label reads supported, which means a
simulation study checked fits of this setup: two events, at least 1,000 patients, the default
models, and times inside the checked range ([how we know the method works](#how-we-know-the-method-works)).
Report each row's own interval. The figure draws each risk as a point with its interval, and the
difference below; nothing is drawn between the two times, because the fit estimates no risk there.
Its brackets mark a band that covers all twelve numbers at once, which no study has checked yet, so
it reads experimental.

The death difference by month 3, -0.005, has an interval from -0.070 to 0.060. The data cannot tell
no change from the true change of -0.059: both lie inside it.

## Months free of readmission and death

The restricted mean survival time is the average number of months a patient spends free of both
events up to the cut-off. Its counterpart for one event, the restricted mean time lost, is the
average number of months spent after that event up to the cut-off.

In [7]:
gained = result.rmst("offered", 3.0, versus="not offered")
saved = result.rmtl("offered", 3.0, "readmission", versus="not offered")
ratio = result["rr_regimen[offered vs not offered, readmission @ t=3]"]
for label, estimate in (
    ("months free of both events in the first 3, offered minus not", gained),
    ("months lost to readmission in the first 3, offered minus not", saved),
    ("risk of readmission by month 3, offered over not offered", ratio),
):
    line = f"{label}: {estimate.psi:.3f} (95% CI {estimate.ci[0]:.3f} to {estimate.ci[1]:.3f})"
    line += f", {estimate.support.label}, true {truth[estimate.name]:.3f}"  # simulation only
    print(line)

months free of both events in the first 3, offered minus not: 0.407 (95% CI 0.249 to 0.565), supported, true 0.556
months lost to readmission in the first 3, offered minus not: -0.393 (95% CI -0.521 to -0.264), supported, true -0.421
risk of readmission by month 3, offered over not offered: 0.606 (95% CI 0.491 to 0.748), supported, true 0.500


Over the first three months, offering navigation to everyone would add 0.407 months free of both
events, about 12 days per patient. The time a patient spends after a first readmission, within
those three months, would change by -0.393 months. The ratio of the two plans' risks of readmission
by month 3, 0.606, says the offer cuts that risk by about two fifths. The difference and the ratio
describe the same effect on two scales: the difference counts patients, and the ratio compares
risks.

## Check the fit

These checks read the fitted models; none can show that the risk score holds every confounder.

In [ ]:
report = result.assess()
checks = report.to_frame()
# Leave out the checks that run only on request, have no formula for this fit, or do not apply.
ran = checks[~checks["status"].isin(["deferred", "unavailable", "not_applicable"])]
print(ran[["check", "status"]].to_string(index=False))
print()
support = result.diagnostics.support()
offer = pd.DataFrame(support.propensity_quantiles).loc[[0.0, 0.01, 0.99, 1.0]]
print("predicted chance of the offer, smallest to largest:")
print(offer.round(3).to_string())
staying = pd.DataFrame(support.censoring_quantiles).loc[[0.0, 0.01, 0.5]]
staying.index = ["smallest", "1%", "50%"]
print("chance of staying in follow-up just before each month, under the offer received:")
print(staying.round(3).to_string())
print(f"share of patient-times the weight floor raised: {support.bounded_share:.3f}")
overlap_fig = plotting.overlap(result, censoring=True)

The nudging step reached its target (`score_equations passed`). A check that reads `completed` ran
and has no pass or fail; its numbers follow. A failed check would show here too. The predicted chance of the offer runs from 0.023 to
0.971, so a few patients had almost no chance of one plan. These chances come from the default
logistic regressions of the offer, and the chances of staying from Cox regressions of leaving
follow-up; both see only the offer and the risk score, so they say nothing about a cause they were
not given. The nudging step weights the patients
who got each plan by one over that chance, so those few carry large weights. The chance of staying
in follow-up until month 3 is at least 0.573 for every patient. The fit raises any chance below a
floor so that no weight grows without limit, and here it raised none (share 0.000). A large share
would mean the estimate leans on a few patients, and is a reason to narrow who the question covers.
Three sensitivity checks that other fits offer have no formula for this kind of fit yet, so the
report lists them as unavailable; the E-value below is the one that applies.

## What if a confounder is missing?

The E-value says how strongly a missing confounder would need to be linked to both the offer and
readmission to explain away the ratio of the two plans' risks by month 3.

In [9]:
evalue = result.sensitivity.evalue(
    ("cif_regimen[offered, readmission @ t=3]", "cif_regimen[not offered, readmission @ t=3]")
)
print(evalue.summary())

                                   E-value
Question:  how strong a confounder      Estimate:    rr_regimen[offered vs not
           would explain away                      offered, readmission @ t=3]
           rr_regimen[offered vs not    Scale:                      risk ratio
           offered, readmission @ t=3]
                             risk ratio    E-value
------------------------------------------------------------------------------
estimate                          0.606       2.69
95% limit nearest the null        0.748       2.01
An unmeasured confounder would need a risk ratio of 2.69 with both the
treatment and the outcome, beyond the measured covariates, to explain away the
estimate; 2.01 to move the interval to the null.
Warnings:
[1] No validation study licenses the E-value of the limit for this kind of
    fit, so read it as descriptive.
Notes:
[2] The E-value is the strength at which the confounder's associations with
    the treatment and with the outcome, taken 

A missing confounder would need to be linked to both the offer and readmission by a risk ratio of
2.69 each, beyond what the risk score explains, to explain away the ratio of 0.606, and by 2.01 to
move its interval to no effect. A weaker link on one side would need a stronger one on the other.
The second number is descriptive, because no study has checked it for this fit. It also assumes the
missing confounder does not change who leaves follow-up. Whether a factor that strong could be
missing is a judgment about the program that the data cannot settle. A stress test that
plants a made-up confounder also runs on this fit; [point-treatment
TMLE](point-treatment-tmle.ipynb) shows how to read one.

## What we can and can't conclude

For the program, had every patient stayed in follow-up: offering navigation to everyone would lower
the risk of readmission by month 3 by about 16 in 100 patients and add about 12 days free of both
events over those months. The data cannot say whether it changes death. These hold for adults
discharged home under the conditions in "Set up the analysis", and lean on the risk score holding
every confounder.

This analysis fits a yes-or-no treatment given once at discharge. It refuses more than two arms,
clusters and sampling weights, which the check-day survival methods take ([what is
refused](../technical-reference/continuous-time-survival.md#what-is-refused)). A treatment that
changes after discharge needs [readmission when death can come
first](longitudinal-survival.ipynb). One simulated sample does not show how the method performs in
general.

## Go deeper

### Refutation

These checks change the data in a known way and refit the readmission difference at month 3, 20
times each:

- `placebo` shuffles the offer among patients, so the difference should be near zero;
- `random_common_cause` adds a pure-noise column, so the difference should barely move;
- `subset` refits on random 70% samples, so the refits should spread about as much as the standard
  error predicts for samples of that size.

None can detect a missing confounder.

In [10]:
# 60 refits, about two and a half minutes.
refutation = result.diagnostics.refute(
    estimand="ate_regimen[offered vs not offered, readmission @ t=3]"
)
print(refutation.summary())

                               Refutation tests
Estimate tested:  ate_regimen[offered   Repeats per test:                   20
                  vs not offered,       Result:                         failed
                  readmission @ t=3] =
                  -0.1596
Tests run:        3
test                     mean         SD   expected           result
------------------------------------------------------------------------------
shuffled treatment    -0.0028     0.0301   about 0            passed
added noise column    -0.1578     0.0029   about -0.1596      no pass or fail
random subsamples     -0.1615     0.0126   SD about 0.02182   failed
Test statistics
test                 statistic                                value   p-value
------------------------------------------------------------------------------
shuffled treatment   t of the refits against 0               -0.410    0.6863
shuffled treatment   mean std. error / refit SD               1.060         -
added noise colu

The shuffled offers give a mean of -0.0028, as expected, and the noise column barely moves the
estimate. The subset test fails: the refits spread less than the standard error predicts (ratio
0.578, where 1 is ideal). If the standard error were wrong in this direction, the interval would be
too wide, which is the safe direction; on your own data, report that the interval may be
conservative. A failure can also happen to a correct fit: the [refutation
study](../technical-reference/method-evidence/refuter-calibration-continuous-survival.md), on fits
like this one without a restricted cut-off, saw the subset test fail in about one fit in eleven with
nothing wrong. A simulation of 100 samples from this page's population, run while writing the page
and not shown here, found the standard error close to the spread of the estimates (0.0338 on
average, against a spread of 0.0327).

### How we know the method works

Simulation studies check each method over many repeated samples. Each uses other simulated data,
with two competing events and 1,000 patients:

| what the page reads | registered study |
| --- | --- |
| each plan's risks and their differences | [continuous-time survival](../technical-reference/method-evidence/continuous-time-survival.md) |
| the mean time free of both events, the time lost to readmission, and the risk ratio | [restricted means and risk ratios](../technical-reference/method-evidence/continuous-survival-functionals.md) |
| the refutation verdicts | [refutation of continuous-time fits](../technical-reference/method-evidence/refuter-calibration-continuous-survival.md) |

No study checks the band over all twelve numbers. The fit can also give a second interval by
refitting on resampled patients (`n_bootstrap=`); it reads `experimental`, and its studies have
not run yet ([the full-refit bootstrap](../technical-reference/continuous-time-survival.md#the-full-refit-bootstrap)).
The 100-sample simulation quoted in this page was run on its own population to check its
readings; the studies above are the method's evidence. The [technical entry](../technical-reference/continuous-time-survival.md#evidence) lists
the method's evidence and limits.

### Counting deaths as dropouts

A common shortcut treats each death like leaving follow-up and fits readmission alone. That asks
another question: the risk of readmission in a world where nobody dies, which no program can bring
about.

In [11]:
from cleverly.datasets import exact_readmission_truth

# The mistake: count each death as leaving follow-up and fit readmission alone.
deaths_as_dropouts = frame.assign(event=np.where(frame["event"].eq(1), 1, 0))
one_event = TimeToEvent(
    time="time",
    event="event",
    treatment="offered",
    baseline=("risk_score",),
    grid="continuous",
    causes={1: "readmission"},
)
one_event_result = CausalStudy(deaths_as_dropouts, design=one_event).estimate(
    RegimeContrast({"offered": 1, "not offered": 0}, reference="not offered", horizons=(1.5, 3.0)),
    random_state=0,
)
death_free = exact_readmission_truth((1.5, 3.0), death_prevented=True)  # simulation only
print("risk of readmission by month 3:")
for plan in ("offered", "not offered"):
    one = one_event_result[f"risk_regimen[{plan} @ t=3]"]
    both = result[f"cif_regimen[{plan}, readmission @ t=3]"]
    print(
        f"  {plan:12s} deaths as dropouts {one.psi:.4f} ({one.ci[0]:.4f} to {one.ci[1]:.4f}),"
        f" true with nobody dying {death_free[f'risk_regimen[{plan} @ t=3]']:.4f}"  # simulation
    )
    print(
        f"  {'':12s} two events         {both.psi:.4f} ({both.ci[0]:.4f} to {both.ci[1]:.4f}),"
        f" true {truth[f'cif_regimen[{plan}, readmission @ t=3]']:.4f}"  # simulation only
    )

risk of readmission by month 3:
  offered      deaths as dropouts 0.2906 (0.2358 to 0.3522), true with nobody dying 0.2357
               two events         0.2452 (0.2054 to 0.2899), true 0.2126
  not offered  deaths as dropouts 0.4669 (0.4110 to 0.5236), true with nobody dying 0.4991
               two events         0.4048 (0.3547 to 0.4568), true 0.4251


With deaths counted as dropouts, the risk of readmission by month 3 is 0.4669 without the offer,
against 0.4048 when death competes. The shortcut is larger because patients who would have died
stay at risk of readmission. Its true value, 0.4991, differs from the program's 0.4251.
For the offered plan, the lower end of the shortcut's interval, 0.2358, sits at its true value,
0.2357. Over the page's 100 simulated samples the same interval contained its value 96 times, so
an interval this close to missing is consistent with chance. The method stays on target in large samples if either the event-rate
models, or the models of the offer and of staying in follow-up, are right. Here the readmission
model is right, while the follow-up model, which now counts deaths, is only approximate. The
question the shortcut answers also assumes that removing death would leave each patient's
readmission rate as it was, which no data can check. Use the risk with both events unless that world is the question.

## Where to go next

[Readmission by day 30, 60 and 90](discharge-to-readmission.ipynb) follows readmission at check
days. [Readmission when death can come first](longitudinal-survival.ipynb) asks the death-free
question on purpose, with treatment at two times. [Plots](plots.ipynb) draws the weight and model
figures of a fit like this one. The user guide page [exact event
times](../user-guide/survival-exact-times.md) lists every option, and the [technical
entry](../technical-reference/continuous-time-survival.md) gives the estimator, its conditions and
what it refuses. The [examples index](index.md) lists every example.